In [ ]:
# Additive bars: independently calibrate and evaluate each bars-per-image condition.
# In Colab, upload margin_dictionary_learning.py before running this cell.
from pathlib import Path
import hashlib
import json
import sys
sys.dont_write_bytecode = True
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
from scipy.optimize import linear_sum_assignment
from scipy.stats import t
from sklearn.decomposition import PCA, FastICA, NMF, DictionaryLearning
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place margin_dictionary_learning.py in the project root or Colab working folder.')
sys.path.insert(0, str(ROOT))
from margin_dictionary_learning import MarginDictionaryLearning, _FISTA

# Experiment settings. The library APIs call the penalty "alpha".
RUN_FITS = False  # False: redraw saved results. True: rerun and replace the CSV.

SIDE, K, N_TRAIN = 6, 12, 50
BARS_CONDITIONS = (3, 4, 5, 6, 7)
FIGURE_BARS = 5  # Preserve the original five-bar illustration.
BARS_PER_IMAGE, NOISE_SD = FIGURE_BARS, 0.10
CALIBRATION_SEEDS = tuple(range(101, 111))
EVALUATION_SEEDS = tuple(range(1, 51))
FIGURE_SEED, N_EXAMPLES = 1, 8
LAMBDA_GRID = [0.03, 0.04, 0.05]
Q_GRID = [0.70, 0.75, 0.80]
N_UPDATES, LEARNING_RATE = 5000, 0.01
CODE_TOL, CODE_MAX_ITER = 1e-5, 5000
REFERENCE_MAX_ITER, RECORD_EVERY = 5000, 100
RECOVERY_THRESHOLD, DEVICE, CPU_THREADS = 0.9, 'cpu', 2
OUTPUT_DIR = ROOT / 'bars_results'
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_PATH = OUTPUT_DIR / 'bars_scores.csv'
ORDINARY, CLIPPED, REFERENCE = 'SDL (ordinary weights)', 'Clipped SDL', 'SDL (scikit-learn)'
SDL_METHODS = (REFERENCE, ORDINARY, CLIPPED)
bar_names = [f'H{i+1}' for i in range(SIDE)] + [f'V{i+1}' for i in range(SIDE)]
torch.set_num_threads(CPU_THREADS)
PLOT_SETTINGS = dict(side=SIDE, atoms=K, observations=N_TRAIN, bars_per_image=BARS_PER_IMAGE,
                     noise=NOISE_SD, calibration_seeds=CALIBRATION_SEEDS,
                     evaluation_seeds=EVALUATION_SEEDS, figure_seed=FIGURE_SEED,
                     examples=N_EXAMPLES, lambda_grid=LAMBDA_GRID, q_grid=Q_GRID,
                     updates=N_UPDATES, lr=LEARNING_RATE, code_tol=CODE_TOL,
                     code_max_iter=CODE_MAX_ITER, reference_max_iter=REFERENCE_MAX_ITER,
                     recovery_threshold=RECOVERY_THRESHOLD)
PLOT_SETTINGS_JSON = json.dumps(PLOT_SETTINGS, sort_keys=True)


def make_bars(seed, n):
    rng = np.random.default_rng(seed)
    images = np.zeros((K, SIDE, SIDE))
    for r in range(SIDE):
        images[r, r, :] = 1
        images[SIDE + r, :, r] = 1
    true_atoms = images.reshape(K, -1) / np.sqrt(SIDE)
    coefficients = np.zeros((n, K))
    for i in range(n):
        indices = rng.choice(K, size=BARS_PER_IMAGE, replace=False)
        coefficients[i, indices] = rng.uniform(0, 1, size=BARS_PER_IMAGE)
    signal = coefficients @ true_atoms
    noise = NOISE_SD * rng.standard_normal((n, SIDE * SIDE))
    noisy = signal + noise
    info = dict(noise_sd=NOISE_SD, min_support=int((coefficients > 0).sum(axis=0).min()),
                snr_db=float(10 * np.log10(np.square(signal).sum() / np.square(noise).sum())),
                negative_pixels_clipped=int((noisy < 0).sum()))
    return noisy.clip(min=0), true_atoms, coefficients, info


def starting_atoms(X, seed):
    indices = np.random.default_rng(seed).choice(len(X), K, replace=False)
    atoms = X[indices].copy()
    return atoms / np.linalg.norm(atoms, axis=1, keepdims=True)


def match_atoms(true_atoms, atoms, sign_invariant=False):
    atoms = np.asarray(atoms, dtype=float)
    unit = atoms / np.maximum(np.linalg.norm(atoms, axis=1, keepdims=True), 1e-12)
    cosine = true_atoms @ unit.T
    similarity = np.abs(cosine) if sign_invariant else cosine
    rows, columns = linear_sum_assignment(-similarity)
    ordered = unit[columns].copy()
    if sign_invariant:
        ordered *= np.where(cosine[rows, columns] < 0, -1, 1)[:, None]
    return ordered, similarity[rows, columns]


def make_weights(q):
    # Pool positive margins across observations and atoms at each update.
    def weights(m):
        if q is None or q == 1:
            return m.clamp_min(0)
        positive = m[m > 0]
        cap = torch.quantile(positive, q) if positive.numel() else m.new_zeros(())
        weights.last_cap = float(cap)
        return m.clamp_min(0).clamp_max(cap)
    weights.last_cap = np.nan
    return weights


def fit_reference(method, X, seed, lam, initial):
    if method == 'PCA':
        model = PCA(n_components=K)
    elif method == 'ICA':
        model = FastICA(n_components=K, whiten='unit-variance', max_iter=2000,
                        tol=1e-4, random_state=seed)
    elif method == 'NMF':
        model = NMF(n_components=K, init='nndsvda', max_iter=2000,
                    tol=1e-4, random_state=seed)
    else:
        model = DictionaryLearning(
            n_components=K, alpha=lam, fit_algorithm='cd',
            transform_algorithm='lasso_cd', transform_alpha=lam,
            positive_code=True, positive_dict=False, max_iter=REFERENCE_MAX_ITER,
            tol=1e-8, random_state=seed,
            dict_init=initial.copy(), code_init=np.zeros((len(X), K)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(X)
    for warning in caught:
        tqdm.write(f'{method}, seed {seed}: {warning.message}')
    atoms = model.mixing_.T if method == 'ICA' else model.components_
    warned = any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return atoms, getattr(model, 'n_iter_', np.nan), warned


def coefficient_certificate(X, atoms, lam):
    # Check coding accuracy at scikit-learn's final dictionary; do not change its atoms.
    x, s = torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(atoms, dtype=torch.float32)
    initial = x.new_zeros((len(X), K))
    solver = _FISTA(x, s, initial, lam, CODE_TOL, CODE_MAX_ITER)
    with torch.no_grad():
        _, _, _, metrics = solver.solve(s, initial)
    return metrics


def fit_key(method, seed, lam, q):
    # At q=1 the clipped rule is exactly ordinary weighting.
    if method == CLIPPED and q == 1:
        method, q = ORDINARY, None
    return method, seed, lam, q


def save_scores():
    if not RUN_FITS:
        raise RuntimeError('Plot-only mode does not modify the score CSV.')
    if records:
        records[0]['plot_settings_json'] = PLOT_SETTINGS_JSON
    for row in records:
        setting = selected.get(row['method'], {})
        row['selected_lambda'] = setting.get('lambda', np.nan)
        row['selected_q'] = setting.get('q', np.nan)
        row['selection_frozen'] = selection_frozen
    temporary = CSV_PATH.with_suffix('.csv.tmp')
    pd.DataFrame(all_records + records).to_csv(temporary, index=False)
    # Windows readers can briefly prevent atomic replacement of the CSV.
    for attempt in range(10):
        try:
            temporary.replace(CSV_PATH)
            break
        except PermissionError:
            if attempt == 9:
                raise
            time.sleep(0.5)


def run_fit(stage, method, seed, lam, q=None):
    if not RUN_FITS:
        raise RuntimeError('Set RUN_FITS=True to fit models.')
    key = fit_key(method, seed, lam, q)
    reused = key in fit_cache
    label = f'{method}, seed={seed}, lambda={lam:g}'
    if q is not None:
        label += f', q={q:g}'
    progress.set_postfix_str(label)
    if reused:
        row, atoms = fit_cache[key]
        row = row.copy()
        row['seconds'] = 0.0
    else:
        X, true_atoms, _, info = data[seed]
        initial = starting_atoms(X, seed)
        begin = time.perf_counter()
        metrics, history, cap, warned = {}, None, np.nan, False
        shared = method in (ORDINARY, CLIPPED)
        if shared:
            def callback(iteration, dictionary):
                _, scores = match_atoms(true_atoms, dictionary)
                progress.set_postfix_str(
                    f'{label}, step={iteration}/{N_UPDATES}, cosine={scores.mean():.4f}')
                return dict(cosine=float(scores.mean()))
            weights = make_weights(q)
            model = MarginDictionaryLearning(
                K, weight_fn=weights, alpha=lam, lr=LEARNING_RATE, n_iter=N_UPDATES,
                code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER,
                dict_init=initial.copy(), code_init=np.zeros((len(X), K)),
                random_state=seed, device=DEVICE, record_every=RECORD_EVERY,
                verbose=False, callback=callback).fit(X)
            atoms, iterations = model.components_, model.n_iter_
            metrics, history = model.final_coding_diagnostics_, model.history_
            cap = weights.last_cap
            assert iterations == N_UPDATES
        else:
            atoms, iterations, warned = fit_reference(method, X, seed, lam, initial)
            if method == REFERENCE:
                metrics = coefficient_certificate(X, atoms, lam)
        if metrics:
            assert metrics['relative_margin_error'] <= CODE_TOL and metrics['relative_gap'] <= CODE_TOL
        _, scores = match_atoms(true_atoms, atoms, method in ('PCA', 'ICA'))
        sdl = method in SDL_METHODS
        limit = N_UPDATES if shared else REFERENCE_MAX_ITER if method == REFERENCE else 2000
        row = dict(
            n_observations=N_TRAIN, side=SIDE, n_components=K,
            active_coefficient_distribution='Uniform[0,1]', bars_per_image=BARS_PER_IMAGE,
            pixel_floor=0.0, last_update_cap=cap,
            mean_cosine=float(scores.mean()), recovered_bars=int((scores >= RECOVERY_THRESHOLD).sum()),
            recovery_threshold=RECOVERY_THRESHOLD, iterations=iterations,
            seconds=time.perf_counter() - begin, convergence_warning=warned,
            hit_iteration_limit=bool(not shared and iterations >= limit),
            max_iter=limit if method != 'PCA' else np.nan,
            n_updates=N_UPDATES if shared else np.nan, lr=LEARNING_RATE if shared else np.nan,
            code_tol=CODE_TOL if sdl else np.nan, code_max_iter=CODE_MAX_ITER if sdl else np.nan,
            device=DEVICE if shared else 'cpu', cpu_threads=CPU_THREADS if sdl else 1,
            initial_dictionary_sha256=hashlib.sha256(initial.tobytes()).hexdigest() if sdl else '',
            relative_margin_error=metrics.get('relative_margin_error', np.nan),
            relative_gap=metrics.get('relative_gap', np.nan),
            **info, **{f'cosine_{name}': float(value) for name, value in zip(bar_names, scores)})
        if history is not None:
            tail = pd.DataFrame(history).query('iteration >= @N_UPDATES - 500')
            row.update(final_atom_change=float(tail.iloc[-1].max_atom_change),
                       cosine_range_last_500=float(tail.cosine.max() - tail.cosine.min()))
        if row['hit_iteration_limit']:
            tqdm.write(f'{label}: reached the iteration limit; inspect convergence before reporting.')
        fit_cache[key] = (row.copy(), np.asarray(atoms).copy())
    row.update(stage=stage, method=method, seed=seed, q=q, reused_fit=reused,
               displayed_in_figure=BARS_PER_IMAGE == FIGURE_BARS and stage != 'calibration'
               and seed == FIGURE_SEED and method != ORDINARY)
    row['lambda'] = lam
    if BARS_PER_IMAGE == FIGURE_BARS and stage != 'calibration' and seed == FIGURE_SEED:
        row['atoms_json'] = json.dumps(np.asarray(atoms).tolist())
        if method == CLIPPED:
            row['observations_json'] = json.dumps(data[seed][0][:N_EXAMPLES].tolist())
            row['truth_json'] = json.dumps(data[seed][1].tolist())
    records.append(row)
    save_scores()
    if BARS_PER_IMAGE == FIGURE_BARS and stage != 'calibration' and seed == FIGURE_SEED:
        fitted_atoms[seed, method] = np.asarray(atoms).copy()
    if not reused:
        progress.update()


def calibration_summary():
    frame = pd.DataFrame(records).query("stage == 'calibration'")
    summary = frame.groupby(['method', 'lambda', 'q'], dropna=False).agg(
        mean_cosine=('mean_cosine', 'mean'), sd_cosine=('mean_cosine', 'std'),
        n=('mean_cosine', 'size')).reset_index()
    assert summary.n.eq(len(CALIBRATION_SEEDS)).all()
    return summary


def choose_settings(summary):
    ranked = summary.sort_values(['mean_cosine', 'lambda', 'q'], ascending=[False, True, False])
    return {method: {'lambda': float(group.iloc[0]['lambda']),
                     'q': float(group.iloc[0]['q']) if method == CLIPPED else None}
            for method, group in ranked.groupby('method')}


def boundary_extensions(settings, lambdas, quantiles, improved):
    new_lambdas = set()
    for method, setting in settings.items():
        if not improved[method]:
            continue
        lower = round(min(lambdas) - 0.01, 12)
        if setting['lambda'] == min(lambdas) and lower > 0:
            new_lambdas.add(lower)
        if setting['lambda'] == max(lambdas):
            new_lambdas.add(round(max(lambdas) + 0.01, 12))
    q = settings[CLIPPED]['q']
    new_q = set()
    if improved[CLIPPED] and q == min(quantiles) and q > 0:
        new_q.add(round(max(0.0, q - 0.05), 10))
    if improved[CLIPPED] and q == max(quantiles) and q < 1:
        new_q.add(round(min(1.0, q + 0.05), 10))
    return sorted(new_lambdas - set(lambdas)), sorted(new_q - set(quantiles))


assert len(set(BARS_CONDITIONS)) == len(BARS_CONDITIONS)
assert all(1 <= count <= K for count in BARS_CONDITIONS) and K == 2 * SIDE
assert FIGURE_BARS in BARS_CONDITIONS
assert set(CALIBRATION_SEEDS).isdisjoint(EVALUATION_SEEDS)
assert FIGURE_SEED in EVALUATION_SEEDS
if RUN_FITS:
    started = time.perf_counter()
    all_records = []
    for BARS_PER_IMAGE in BARS_CONDITIONS:
        PLOT_SETTINGS['bars_per_image'] = BARS_PER_IMAGE
        PLOT_SETTINGS_JSON = json.dumps(PLOT_SETTINGS, sort_keys=True)
        data = {seed: make_bars(seed, N_TRAIN) for seed in CALIBRATION_SEEDS + EVALUATION_SEEDS}
        records, fitted_atoms, fit_cache, completed, selected = [], {}, {}, set(), {}
        selection_frozen = False
        lambdas, quantiles = sorted(set(LAMBDA_GRID)), sorted(set(Q_GRID))
        initial_count = len(CALIBRATION_SEEDS) * len(lambdas) * (2 + sum(q != 1 for q in quantiles))
        print(f'CPU | {SIDE}x{SIDE}, {K} bars, N={N_TRAIN}, {BARS_PER_IMAGE} bars/image, noise SD={NOISE_SD}.')
        print(f'Calibration seeds: {list(CALIBRATION_SEEDS)}; evaluation seeds: {list(EVALUATION_SEEDS)}; figure seed: {FIGURE_SEED}.')
        print(f'lambda grid={lambdas}; q grid={quantiles}; {N_UPDATES} dictionary updates per shared-solver fit.')
        figure_count = 3 if BARS_PER_IMAGE == FIGURE_BARS else 0
        print(f'{initial_count} initial calibration fits, then up to {3 * len(EVALUATION_SEEDS)} evaluation fits and {figure_count} figure fits.')
        print('Each condition starts its own calibration grid; progress bars report phase timing.')

        with threadpool_limits(limits=CPU_THREADS):
            round_index, previous_best = 0, None
            while True:
                jobs = [(method, seed, lam, q)
                        for method in SDL_METHODS for lam in lambdas
                        for q in (quantiles if method == CLIPPED else [None])
                        for seed in CALIBRATION_SEEDS
                        if (method, seed, lam, q) not in completed]
                new_count = len({fit_key(*job) for job in jobs} - fit_cache.keys())
                print(f'Search round {round_index + 1}: {new_count} new fits; lambda={lambdas}; q={quantiles}.')
                with tqdm(total=new_count, desc=f'Calibration {round_index + 1}', unit='fit') as progress:
                    for method, seed, lam, q in jobs:
                        run_fit('calibration', method, seed, lam, q)
                        completed.add((method, seed, lam, q))
                summary = calibration_summary()
                selected = choose_settings(summary)
                print('Calibration mean cosine, Clipped SDL:')
                print(summary[summary.method == CLIPPED].pivot(index='lambda', columns='q', values='mean_cosine').to_string(float_format=lambda x: f'{x:.5f}'))
                print('Calibration results, ordinary weights and scikit-learn:')
                print(summary[summary.method != CLIPPED].drop(columns='q').to_string(index=False, float_format=lambda x: f'{x:.5f}'))
                print('Selected settings:')
                for method in SDL_METHODS:
                    setting = selected[method]
                    q_text = f", q={setting['q']:g}" if method == CLIPPED else ''
                    print(f"  {method}: lambda={setting['lambda']:g}{q_text}")
                best = summary.groupby('method').mean_cosine.max().to_dict()
                # Try an edge once, then continue only while that method's best score improves.
                improved = {method: previous_best is None or best[method] > previous_best[method]
                            for method in SDL_METHODS}
                extra_lambdas, extra_q = boundary_extensions(selected, lambdas, quantiles, improved)
                save_scores()
                if not extra_lambdas and not extra_q:
                    selection_frozen = True
                    save_scores()
                    break
                print(f'Expanding: lambda={extra_lambdas}, q={extra_q}.')
                lambdas = sorted(set(lambdas + extra_lambdas))
                quantiles = sorted(set(quantiles + extra_q))
                previous_best = best
                round_index += 1

            print('No improving boundary remains. Selections frozen; starting independent evaluation.')
            jobs = [(method, seed, selected[method]['lambda'], selected[method]['q'])
                    for seed in EVALUATION_SEEDS for method in SDL_METHODS]
            new_count = len({fit_key(*job) for job in jobs} - fit_cache.keys())
            with tqdm(total=new_count, desc='Evaluation', unit='fit') as progress:
                for method, seed, lam, q in jobs:
                    run_fit('evaluation', method, seed, lam, q)
            if BARS_PER_IMAGE == FIGURE_BARS:
                with threadpool_limits(limits=1):
                    with tqdm(total=3, desc='Figure models', unit='fit') as progress:
                        for method in ('PCA', 'ICA', 'NMF'):
                            run_fit('figure', method, FIGURE_SEED, np.nan)
        all_records.extend(records)
        if BARS_PER_IMAGE == FIGURE_BARS:
            illustration_data, illustration_atoms = data, fitted_atoms
    saved = pd.DataFrame(all_records)
    data, fitted_atoms = illustration_data, illustration_atoms

else:
    started = time.perf_counter()
    if not CSV_PATH.exists():
        raise FileNotFoundError('No saved results; set RUN_FITS=True to generate them.')
    saved = pd.read_csv(CSV_PATH)
    required = {'atoms_json', 'observations_json', 'truth_json', 'plot_settings_json'}
    if not required.issubset(saved.columns):
        raise ValueError('This CSV lacks figure data; set RUN_FITS=True to regenerate it.')

# Validate each condition independently, including its frozen parameter selection.
# The original five-bar CSV can still redraw the illustration before the sweep is run.
evaluation = saved[saved.stage == 'evaluation']
available_conditions = tuple(sorted(saved.bars_per_image.unique()))
if set(available_conditions) not in (set(BARS_CONDITIONS), {FIGURE_BARS}):
    raise ValueError('The saved sweep is incomplete or has different conditions; set RUN_FITS=True.')
selected_by_condition = {}
for count in available_conditions:
    condition = saved[saved.bars_per_image == count]
    specifications = condition.plot_settings_json.dropna().unique()
    expected_settings = dict(PLOT_SETTINGS, bars_per_image=int(count))
    if len(specifications) != 1 or json.loads(specifications[0]) != json.loads(json.dumps(expected_settings)):
        raise ValueError(f'Settings differ for {count} bars/image; restore them or set RUN_FITS=True.')
    fits = evaluation[evaluation.bars_per_image == count]
    expected = {(seed, method) for seed in EVALUATION_SEEDS for method in SDL_METHODS}
    assert len(fits) == len(expected) and set(zip(fits.seed, fits.method)) == expected
    assert fits.selection_frozen.eq(True).all()
    assert np.isfinite(fits.mean_cosine).all()
    assert fits.groupby('seed').initial_dictionary_sha256.nunique().eq(1).all()
    selected_by_condition[count] = {}
    for method, rows in fits.groupby('method'):
        assert rows.selected_lambda.nunique() == 1 and rows.selected_q.nunique(dropna=False) == 1
        first = rows.iloc[0]
        assert rows['lambda'].eq(first.selected_lambda).all()
        if method == CLIPPED:
            assert rows.q.eq(first.selected_q).all()
        selected_by_condition[count][method] = {
            'lambda': float(first.selected_lambda),
            'q': float(first.selected_q) if method == CLIPPED else None}

if not RUN_FITS:
    figure = saved[(saved.bars_per_image == FIGURE_BARS) & (saved.seed == FIGURE_SEED)
                   & saved.stage.isin(['evaluation', 'figure'])]
    assert not figure.method.duplicated().any()
    fitted_atoms = {(FIGURE_SEED, row.method): np.asarray(json.loads(row.atoms_json), dtype=float)
                    for row in figure.itertuples()}
    source = figure[figure.method == CLIPPED].iloc[0]
    data = {FIGURE_SEED: (np.asarray(json.loads(source.observations_json)),
                         np.asarray(json.loads(source.truth_json)), None, {})}
    print('Saved results loaded; no fits or CSV writes.')

results = saved
summary = evaluation.groupby(['bars_per_image', 'method']).agg(
    mean_cosine=('mean_cosine', 'mean'), sd_cosine=('mean_cosine', 'std'),
    n=('mean_cosine', 'size'), mean_bars=('recovered_bars', 'mean'),
    sd_bars=('recovered_bars', 'std'), total_bars=('recovered_bars', 'sum'))
print(f'Final recovery: mean +/- sample SD across {len(EVALUATION_SEEDS)} datasets per condition.')
for count in available_conditions:
    print(f'{count} bars per image:')
    for method in SDL_METHODS:
        row, setting = summary.loc[count, method], selected_by_condition[count][method]
        q_text = f", q={setting['q']:g}" if method == CLIPPED else ''
        print(f"  {method}: lambda={setting['lambda']:g}{q_text}, "
              f'cosine={row.mean_cosine:.5f} +/- {row.sd_cosine:.5f}; '
              f'bars >= {RECOVERY_THRESHOLD:g}: {row.mean_bars:.2f} +/- {row.sd_bars:.2f} '
              f'({int(row.total_bars)}/{K * len(EVALUATION_SEEDS)} total).')
    paired = evaluation[evaluation.bars_per_image == count].pivot(
        index='seed', columns='method', values='mean_cosine')
    for baseline in (ORDINARY, REFERENCE):
        difference = paired[CLIPPED] - paired[baseline]
        wins, ties, losses = (int((difference > 1e-8).sum()), int((difference.abs() <= 1e-8).sum()),
                              int((difference < -1e-8).sum()))
        half_width = t.ppf(.975, len(difference) - 1) * difference.std(ddof=1) / np.sqrt(len(difference))
        print(f'  Clipped minus {baseline}: {difference.mean():+.5f}; '
              f'pointwise 95% CI [{difference.mean()-half_width:+.5f}, {difference.mean()+half_width:+.5f}]; '
              f'wins/ties/losses={wins}/{ties}/{losses}.')


X, true_atoms, _, info = data[FIGURE_SEED]
plt.rcParams.update({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix',
                     'font.size': 8, 'axes.linewidth': 0.5})
plot_rows = [
    ('(a) observations', None), ('(b) generating bars', None),
    ('(c) PCA', 'PCA'), ('(d) ICA', 'ICA'), ('(e) NMF', 'NMF'),
    ('(f) SDL (scikit-learn)', 'SDL (scikit-learn)'),
    ('(g) Clipped SDL', 'Clipped SDL'),
]
fig = plt.figure(figsize=(7.4, 4.4))
grid = GridSpec(len(plot_rows), 2 * K, figure=fig, height_ratios=[1.4] + [1] * (len(plot_rows) - 1),
                left=0.17, right=0.99, top=0.98, bottom=0.04, hspace=0.65, wspace=0.18)
observation_limit = float(np.abs(X[:N_EXAMPLES]).max())
for r, (label, method) in enumerate(plot_rows):
    if r == 0:
        images, scores, width = X[:N_EXAMPLES], None, 3
    elif r == 1:
        images, scores, width = true_atoms, None, 2
    else:
        images, scores = match_atoms(true_atoms, fitted_atoms[FIGURE_SEED, method], method in ('PCA', 'ICA'))
        width = 2
    for j, pixels in enumerate(images):
        ax = fig.add_subplot(grid[r, width * j:width * (j + 1)])
        limit = observation_limit if r == 0 else max(float(np.abs(pixels).max()), 1e-12)
        ax.imshow(pixels.reshape(SIDE, SIDE), cmap='RdBu_r', vmin=-limit, vmax=limit,
                  interpolation='nearest')
        ax.set(xticks=[], yticks=[])
        for spine in ax.spines.values():
            spine.set_color('0.55')
            spine.set_linewidth(0.4)
        if scores is not None:
            ax.set_xlabel(f'{scores[j]:.3f}', labelpad=1.5, fontsize=6)
        if r == 1:
            ax.set_title(bar_names[j], fontsize=6, pad=2)
        if j == 0:
            ax.annotate(label, xy=(0, 0.5), xycoords='axes fraction', xytext=(-6, 0),
                        textcoords='offset points', ha='right', va='center', fontsize=7)
fig.savefig(OUTPUT_DIR / 'fig_dl_bars.pdf', bbox_inches='tight')
plt.show()
print(f'Results: {CSV_PATH}; illustration: {OUTPUT_DIR / "fig_dl_bars.pdf"}.')

if set(available_conditions) == set(BARS_CONDITIONS):
    # Error bars: pointwise 95% t intervals for the mean across evaluation seeds.
    # Method colours are identified in the thesis caption, as in the other figures.
    with plt.rc_context({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix',
                         'font.size': 10, 'axes.linewidth': .6,
                         'axes.spines.top': False, 'axes.spines.right': False,
                         'pdf.fonttype': 42, 'ps.fonttype': 42}):
        fig, ax = plt.subplots(figsize=(5.4, 3.25))
        colors = {CLIPPED: '#2678AC', ORDINARY: '#C47A44', REFERENCE: '#657383'}
        markers = {CLIPPED: 'o', ORDINARY: 's', REFERENCE: '^'}
        offsets = {CLIPPED: .06, ORDINARY: 0, REFERENCE: -.06}
        for method in SDL_METHODS:
            values = summary.xs(method, level='method').loc[list(BARS_CONDITIONS)]
            half_width = t.ppf(.975, values.n - 1) * values.sd_cosine / np.sqrt(values.n)
            ax.errorbar(np.asarray(BARS_CONDITIONS) + offsets[method], values.mean_cosine,
                        yerr=half_width, color=colors[method], marker=markers[method],
                        linewidth=1.3, markersize=4, capsize=3, elinewidth=.9)
        ax.set(xlabel='Bars per image', ylabel='Mean signed matched cosine',
               xticks=BARS_CONDITIONS, xlim=(min(BARS_CONDITIONS)-.35, max(BARS_CONDITIONS)+.35))
        ax.set_axisbelow(True)
        ax.grid(axis='y', color='#E8EBEE', linewidth=.6)
        ax.margins(y=.12)
        fig.tight_layout(pad=.7)
        fig.savefig(OUTPUT_DIR / 'fig_dl_bars_recovery.pdf', bbox_inches='tight', pad_inches=.05)
        plt.show()
    print('Recovery figure: blue circles = clipped; orange squares = ordinary; grey triangles = scikit-learn.')
    print(OUTPUT_DIR / 'fig_dl_bars_recovery.pdf')
else:
    print('Only the original five-bar results are available. The sensitivity figure requires RUN_FITS=True.')

print(f'Finished in {time.perf_counter() - started:.1f}s.')
